# Evaluating Grok Applications: Deterministic Checks + LLM-as-Judge

"It looked good when I tried it" is not an evaluation. The moment you change a prompt, swap a model
version, or raise the temperature, you need a way to answer one question: **did this get better or
worse?**

This cookbook builds a small but complete evaluation harness for a Grok application. The key idea is a
**two-tier judging strategy**:

| Tier | Cost | Use it for |
|------|------|-----------|
| **1. Deterministic checks** | 0 tokens, instant | anything with a right answer — format, schema, required content, forbidden content, latency |
| **2. LLM-as-judge (Grok)** | 1 call per case | genuinely subjective qualities — tone, helpfulness, faithfulness |

Run tier 1 first and let it **short-circuit**: if an output isn't even valid JSON, don't spend a judge
call asking whether its tone is friendly. Most regressions are caught for free, and you only pay the
model where human-like judgment is genuinely required.

**What you'll build**

1. An **eval dataset** — cases with inputs and expectations.
2. **Deterministic graders** — schema, required/forbidden terms, exact match.
3. An **LLM judge** on Grok with a strict rubric and structured output.
4. A **scored run** with a pass/fail gate and a **regression comparison** between two variants.


## Setup

We use the same OpenAI-compatible client as the rest of the cookbook. The deterministic graders are pure Python, so the whole harness runs even without an API key — the app and judge then fall back to clearly-labeled offline stand-ins.

In [ ]:
%pip install openai pydantic python-dotenv --quiet

In [2]:
import os, json, re, time
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
XAI_API_KEY = os.environ.get("XAI_API_KEY", "")
MODEL = "grok-4"
LIVE = bool(XAI_API_KEY)
client = OpenAI(base_url="https://api.x.ai/v1", api_key=XAI_API_KEY) if LIVE else None
print("Mode:", "LIVE (real Grok API)" if LIVE else "OFFLINE DEMO (deterministic graders run for real)")

Mode: OFFLINE DEMO (deterministic graders run for real)


## Step 1 — The eval dataset

An eval set is just cases: an input, plus what a good answer must satisfy. Keep expectations
**machine-checkable** wherever you can — that's what makes tier 1 free.

Our toy application is a support-ticket classifier that must return strict JSON.

In [3]:
EVAL_CASES = [
    {
        "id": "billing_refund",
        "input": "I was charged twice for my subscription this month. I want one charge refunded.",
        "expect_category": "billing",
        "must_include": ["refund"],
        "must_not_include": ["password"],
    },
    {
        "id": "login_2fa",
        "input": "I can't log in, the two-factor code never arrives on my phone.",
        "expect_category": "account_access",
        "must_include": ["login"],
        "must_not_include": ["refund"],
    },
    {
        "id": "bug_export",
        "input": "Exporting a CSV crashes the dashboard every time I click export.",
        "expect_category": "bug",
        "must_include": ["export"],
        "must_not_include": ["refund"],
    },
]
print(f"{len(EVAL_CASES)} eval cases")

3 eval cases


## Step 2 — The application under test

The thing we're evaluating: a classifier prompted to return strict JSON with a category, a summary, and
a suggested reply. In offline mode we return two *variants* of canned outputs so the regression
comparison later has something real to compare — variant `b` deliberately has a defect.

In [4]:
SYSTEM = (
    "You triage support tickets. Reply with ONLY a JSON object: "
    '{"category": one of ["billing","account_access","bug","other"], '
    '"summary": string, "reply": string}. No prose outside the JSON.'
)

OFFLINE = {
    "a": {  # the good variant
        "billing_refund": '{"category":"billing","summary":"Duplicate charge this month","reply":"Sorry about that - I have issued a refund for the duplicate charge."}',
        "login_2fa": '{"category":"account_access","summary":"2FA code not arriving","reply":"Let us fix your login - I can resend the two-factor code."}',
        "bug_export": '{"category":"bug","summary":"CSV export crashes dashboard","reply":"Thanks for reporting - the export crash is now with our engineers."}',
    },
    "b": {  # the regressed variant: one wrong category + one non-JSON answer
        "billing_refund": '{"category":"other","summary":"Customer mentions a charge","reply":"We have received your refund request."}',
        "login_2fa": "Sure! I can help you with your login problem.",  # not JSON at all
        "bug_export": '{"category":"bug","summary":"CSV export crashes dashboard","reply":"Thanks for reporting - the export crash is now with our engineers."}',
    },
}


def run_app(case: dict, variant: str = "a", temperature: float = 0.0) -> str:
    if LIVE:
        resp = client.chat.completions.create(
            model=MODEL, temperature=temperature,
            messages=[{"role": "system", "content": SYSTEM},
                      {"role": "user", "content": case["input"]}])
        return resp.choices[0].message.content
    return OFFLINE[variant][case["id"]]


print(run_app(EVAL_CASES[0]))

{"category":"billing","summary":"Duplicate charge this month","reply":"Sorry about that - I have issued a refund for the duplicate charge."}


## Step 3 — Tier 1: deterministic graders (zero tokens)

These are the checks with an objectively right answer. They cost nothing, never flake, and catch the
majority of real regressions: broken JSON, wrong label, missing required content, leaked forbidden
content.

Each grader returns `(passed, detail)` so failures are self-explaining in the report.

In [5]:
def grade_valid_json(output: str, case: dict) -> tuple[bool, str]:
    try:
        json.loads(output)
        return True, "valid JSON"
    except json.JSONDecodeError as e:
        return False, f"invalid JSON: {e.msg}"


def grade_schema(output: str, case: dict) -> tuple[bool, str]:
    try:
        obj = json.loads(output)
    except json.JSONDecodeError:
        return False, "unparseable"
    required = {"category", "summary", "reply"}
    missing = required - obj.keys()
    if missing:
        return False, f"missing keys: {sorted(missing)}"
    if obj["category"] not in {"billing", "account_access", "bug", "other"}:
        return False, f"category not in enum: {obj['category']!r}"
    return True, "schema ok"


def grade_category(output: str, case: dict) -> tuple[bool, str]:
    try:
        got = json.loads(output).get("category")
    except json.JSONDecodeError:
        return False, "unparseable"
    want = case["expect_category"]
    return (got == want), f"expected {want!r}, got {got!r}"


def grade_content(output: str, case: dict) -> tuple[bool, str]:
    low = output.lower()
    for term in case.get("must_include", []):
        if term not in low:
            return False, f"missing required term {term!r}"
    for term in case.get("must_not_include", []):
        if term in low:
            return False, f"contains forbidden term {term!r}"
    return True, "content ok"


DETERMINISTIC = [grade_valid_json, grade_schema, grade_category, grade_content]

# Demo on one good and one broken output:
for label, out in [("good", OFFLINE["a"]["login_2fa"]), ("regressed", OFFLINE["b"]["login_2fa"])]:
    print(f"{label}:")
    for g in DETERMINISTIC:
        ok, detail = g(out, EVAL_CASES[1])
        print(f"   {'PASS' if ok else 'FAIL'}  {g.__name__:<18} {detail}")

good:
   PASS  grade_valid_json   valid JSON
   PASS  grade_schema       schema ok
   PASS  grade_category     expected 'account_access', got 'account_access'
   PASS  grade_content      content ok
regressed:
   FAIL  grade_valid_json   invalid JSON: Expecting value
   FAIL  grade_schema       unparseable
   FAIL  grade_category     unparseable
   PASS  grade_content      content ok


## Step 4 — Tier 2: LLM-as-judge on Grok

Some qualities have no regex: is the reply *actually helpful*? Is the tone right for a frustrated
customer? For these we ask Grok to score the output against a **strict rubric** and return structured
JSON.

Three practices keep judges trustworthy:

* **A narrow rubric with explicit anchors** — vague criteria produce vague, drifting scores.
* **`temperature=0` and structured output** — the judge should be as reproducible as we can make it.
* **Require a reason** — a score without a justification is unreviewable, and reading the reasons is how
  you discover the judge itself is wrong.

In [6]:
JUDGE_SYSTEM = (
    "You are a strict evaluator. Score the assistant's support reply on two criteria, 1-5:\n"
    "  helpfulness: 5 = directly resolves the user's problem; 3 = partially useful; 1 = no real help.\n"
    "  tone: 5 = warm and professional; 3 = neutral/robotic; 1 = dismissive or rude.\n"
    'Reply with ONLY JSON: {"helpfulness": int, "tone": int, "reason": string}.'
)


def llm_judge(ticket: str, reply: str) -> dict:
    if LIVE:
        resp = client.chat.completions.create(
            model=MODEL, temperature=0,
            messages=[{"role": "system", "content": JUDGE_SYSTEM},
                      {"role": "user", "content": f"Ticket: {ticket}\n\nAssistant reply: {reply}"}])
        try:
            return json.loads(resp.choices[0].message.content)
        except json.JSONDecodeError:
            return {"helpfulness": 0, "tone": 0, "reason": "judge returned non-JSON"}
    # OFFLINE DEMO: a scripted judgement so the harness runs end-to-end.
    warm = any(w in reply.lower() for w in ["sorry", "thanks", "let us", "happy"])
    return {"helpfulness": 4 if len(reply) > 40 else 2,
            "tone": 5 if warm else 3,
            "reason": "offline demo heuristic stand-in for a real Grok judgement"}


print(llm_judge(EVAL_CASES[0]["input"], json.loads(OFFLINE["a"]["billing_refund"])["reply"]))

{'helpfulness': 4, 'tone': 5, 'reason': 'offline demo heuristic stand-in for a real Grok judgement'}


## Step 5 — The harness: short-circuit, score, gate

Now the two tiers combine. Deterministic checks run first; **if any fails, we skip the judge entirely** —
there's no point paying to rate the tone of malformed output. That single rule is what keeps an eval
suite cheap as it grows.

We gate on two thresholds: deterministic checks must pass 100% (they're objective), and judged scores
must clear a quality bar.

In [7]:
JUDGE_THRESHOLD = 3.5


def evaluate(variant: str, cases=EVAL_CASES) -> dict:
    rows, judge_calls = [], 0
    for case in cases:
        t0 = time.perf_counter()
        output = run_app(case, variant=variant)
        latency_ms = (time.perf_counter() - t0) * 1000

        failures = []
        for g in DETERMINISTIC:
            ok, detail = g(output, case)
            if not ok:
                failures.append(f"{g.__name__}: {detail}")

        judged = None
        if not failures:  # short-circuit: only judge outputs that are objectively well-formed
            reply = json.loads(output)["reply"]
            judged = llm_judge(case["input"], reply)
            judge_calls += 1

        rows.append({"case": case["id"], "failures": failures, "judged": judged,
                     "latency_ms": round(latency_ms, 1)})

    det_pass = sum(1 for r in rows if not r["failures"])
    scores = [ (r["judged"]["helpfulness"] + r["judged"]["tone"]) / 2
               for r in rows if r["judged"] ]
    avg = round(sum(scores) / len(scores), 2) if scores else 0.0
    return {"variant": variant, "rows": rows, "det_pass": det_pass, "total": len(cases),
            "avg_score": avg, "judge_calls": judge_calls}


def report(res: dict) -> None:
    print(f"=== variant {res['variant']} ===")
    print(f"deterministic: {res['det_pass']}/{res['total']} passed | "
          f"judge calls: {res['judge_calls']} (saved {res['total'] - res['judge_calls']}) | "
          f"avg judged score: {res['avg_score']}")
    for r in res["rows"]:
        status = "PASS" if not r["failures"] else "FAIL"
        extra = "" if r["failures"] else f" score={(r['judged']['helpfulness'] + r['judged']['tone']) / 2}"
        print(f"  {status} {r['case']:<16}{extra}")
        for f in r["failures"]:
            print(f"        - {f}")
    gate = res["det_pass"] == res["total"] and res["avg_score"] >= JUDGE_THRESHOLD
    print("GATE:", "PASS ✅" if gate else "FAIL ❌")


baseline = evaluate("a")
report(baseline)

=== variant a ===
deterministic: 3/3 passed | judge calls: 3 (saved 0) | avg judged score: 4.5
  PASS billing_refund   score=4.5
  PASS login_2fa        score=4.5
  PASS bug_export       score=4.5
GATE: PASS ✅


## Step 6 — Regression comparison

The harness pays for itself when you change something. Run the same cases against a second variant and
diff the results — this is the check you put in CI so a prompt tweak can't silently degrade quality.

In [8]:
candidate = evaluate("b")
report(candidate)

print("\n=== baseline vs candidate ===")
print(f"deterministic: {baseline['det_pass']}/{baseline['total']} -> {candidate['det_pass']}/{candidate['total']}")
print(f"avg judged:    {baseline['avg_score']} -> {candidate['avg_score']}")
regressions = [r["case"] for r in candidate["rows"] if r["failures"]]
if regressions:
    print("REGRESSED cases:", regressions)
    print("VERDICT: do not ship ❌")
else:
    print("VERDICT: no regressions ✅")

=== variant b ===
deterministic: 1/3 passed | judge calls: 1 (saved 2) | avg judged score: 4.5
  FAIL billing_refund  
        - grade_category: expected 'billing', got 'other'
  FAIL login_2fa       
        - grade_valid_json: invalid JSON: Expecting value
        - grade_schema: unparseable
        - grade_category: unparseable
  PASS bug_export       score=4.5
GATE: FAIL ❌

=== baseline vs candidate ===
deterministic: 3/3 -> 1/3
avg judged:    4.5 -> 4.5
REGRESSED cases: ['billing_refund', 'login_2fa']
VERDICT: do not ship ❌


## Recap

An evaluation harness doesn't have to be heavy to be useful:

1. **Write cases with machine-checkable expectations** — then most grading is free.
2. **Tier 1: deterministic graders** catch format, schema, label, and content errors at zero token cost.
3. **Tier 2: an LLM judge on Grok** rates only what genuinely needs judgment — with a strict rubric,
   `temperature=0`, and a required reason.
4. **Short-circuit** — never spend a judge call on output that already failed an objective check.
5. **Compare variants** to turn "seems better" into a number, and gate your releases on it.

**Next steps:** set `XAI_API_KEY` (see `.env.example`) and re-run to evaluate live Grok output with a
real Grok judge. From there: add cases every time you find a bug (that's how the suite grows where it
matters), and wire the gate into CI.